In [10]:
df = spark.read.format("delta").table("silver_health_indicators")

StatementMeta(, ce7ec2a1-64c6-4bfe-ba04-0ecc052ae016, 12, Finished, Available, Finished, False)

In [11]:
print(df.count())
print(len(df.columns))

StatementMeta(, ce7ec2a1-64c6-4bfe-ba04-0ecc052ae016, 13, Finished, Available, Finished, False)

83908
9


In [13]:
from pyspark.sql.functions import monotonically_increasing_id

# dim_indicator
dim_indicator = df.select('indicator', 'type').distinct()
dim_indicator = dim_indicator.withColumn('indicator_key', monotonically_increasing_id())
dim_indicator.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("dim_indicator")

# dim_geography
dim_geography = df.select('province').distinct()
dim_geography = dim_geography.withColumn('geography_key', monotonically_increasing_id())
dim_geography.write.format("delta").mode("overwrite").saveAsTable("dim_geography")

# dim_date
dim_date = df.select('year').distinct()
dim_date = dim_date.withColumn('date_key', monotonically_increasing_id())
dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")

StatementMeta(, ce7ec2a1-64c6-4bfe-ba04-0ecc052ae016, 15, Finished, Available, Finished, False)

In [14]:
fact = df \
    .join(dim_indicator, on='indicator', how='left') \
    .join(dim_geography, on='province', how='left') \
    .join(dim_date, on='year', how='left') \
    .select('indicator_key', 'geography_key', 'date_key', 
            'age_group', 'sex', 'characteristic', 'unit_of_measure', 'value')

fact.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("fact_health_indicators")

StatementMeta(, ce7ec2a1-64c6-4bfe-ba04-0ecc052ae016, 16, Finished, Available, Finished, False)